In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn.preprocessing
from tqdm import tqdm
import time

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FORECAST_HORIZON = 28

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout
        )
        # self.fc = nn.Linear(hidden_size, 1)
        self.fc = nn.Linear(hidden_size, FORECAST_HORIZON)

    def forward(self, x):
        device = DEVICE
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.to(device)

In [3]:
class DatasetProcessor(torch.utils.data.Dataset):
    def __init__(self, df, feature_cols, seq_len, target_col_idx, scaler):
        self.seq_len = seq_len
        self.target_col = target_col_idx
        self.groups = []
        self.samples = []

        for group_key, group_df in tqdm(df.groupby(['item_id', 'store_id']), desc="Indexing groups"):
            group_values = scaler.transform(group_df[feature_cols].values)
            group_idx = len(self.groups)
            self.groups.append(group_values.astype(np.float32))
            # for i in range(len(group_values) - seq_len):
            #     self.samples.append((group_idx, i))
            for i in range(len(group_values) - seq_len - FORECAST_HORIZON + 1):
                self.samples.append((group_idx, i))

        self.groups = [torch.tensor(g).share_memory_() for g in self.groups]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        group_idx, i = self.samples[idx]
        group_values = self.groups[group_idx]
        X = group_values[i : i + self.seq_len, :]
        # y = group_values[i + self.seq_len, self.target_col]
        y = group_values[i + self.seq_len : i + self.seq_len + FORECAST_HORIZON, self.target_col]
        return X, y

In [ ]:
dataset_file = 'final_improved_dataset_reduced.pqt'

from google.colab import drive
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Master training')

df = pd.read_parquet(f'/content/drive/MyDrive/Colab Notebooks/Master training/Data/{dataset_file}')

df.head(5)

In [5]:
train_mask = ~(
    ((df['year'] == 2016) & (df['month'] == 4) & (df['day_of_month'] >= 25)) |
    ((df['year'] == 2016) & (df['month'] == 5))
)

df_train = df[train_mask]

lookback_mask = (
    ((df['year'] == 2016) & (df['month'] == 3) & (df['day_of_month'] >= 1))
)
df_eval = df[lookback_mask | ~train_mask]

In [ ]:
sequence_length = 28

scaler = sklearn.preprocessing.MinMaxScaler()
feature_cols = [
    'units_sold',
    'item_id', 'store_id', 'item_category', 'sell_price', 'consumer_sentiment',
    'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'state_snap',
    'state_cpi', 'state_ur', 'state_gas_price',
    'state_cpi_mom_delta', 'state_ur_mom_delta', 'state_gas_wow_delta',
    'day_of_week', 'day_of_month', 'month', 'year'
]

scaler.fit(df_train[feature_cols].values)

units_col = feature_cols.index('units_sold')

dataset_train = DatasetProcessor(df_train, feature_cols, sequence_length, units_col, scaler)
dataset_eval = DatasetProcessor(df_eval, feature_cols, sequence_length, units_col, scaler)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = LSTM(
    input_size=21,
    hidden_size=256,
    num_layers=2,
    dropout=0.1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)


loader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=32768, shuffle=True,
    num_workers=16, pin_memory=True,
    prefetch_factor=4, persistent_workers=True)

loader_eval = torch.utils.data.DataLoader(
    dataset_eval, batch_size=32768, shuffle=False,
    num_workers=16, pin_memory=True,
    prefetch_factor=4, persistent_workers=True)

In [ ]:
from torch.amp import autocast, GradScaler
scaler_amp = GradScaler(device='cuda')


history = {
    'train_loss': [],
    'val_loss': [],
    'epoch': [],
    'epoch_time': []
}

num_epochs = 10
best_loss = float('inf')

for epoch in range(num_epochs):
    # training
    model.train()
    epoch_loss = 0.0
    start_time = time.time()

    for x_batch, y_batch in loader_train:
      x_batch = x_batch.to(device)
      y_batch = y_batch.to(device)

      optimizer.zero_grad()

      with autocast('cuda'):
          output = model(x_batch).squeeze(-1)
          loss = loss_fn(output, y_batch)

      scaler_amp.scale(loss).backward()
      scaler_amp.step(optimizer)
      scaler_amp.update()

      epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(loader_train)

    # validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in loader_eval:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            output = model(x_batch).squeeze(-1)
            loss = loss_fn(output, y_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(loader_eval)
    scheduler.step(avg_val_loss)
    epoch_time = time.time() - start_time


    # save model at checkpoint
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'epoch': epoch,
        }, f'/content/drive/MyDrive/Colab Notebooks/Master training/LSTM_checkpoint.pth')
        print(f"New checkpoint saved - (val_loss: {best_loss:.10f})")

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['epoch'].append(epoch)
    history['epoch_time'].append(epoch_time)

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.10f} | Val Loss: {avg_val_loss:.10f} | Time: {epoch_time:.1f}s")

print("Training done.")

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'epoch': num_epochs,
}, f'/content/drive/MyDrive/Colab Notebooks/Master training/LSTM_trained.pth')